# 8.2 Coding Practice: Recommendation Systems and Personalized Text Ranking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module8/week08_recommendation_practice.ipynb)

This notebook continues the reading's running example. **Alex** knows Python and SQL and wants an early-career reporting role. All job descriptions, ratings, and relevance judgments below are teaching examples, not real postings or measured outcomes.

You will build the same job collection into three recommendation approaches:

1. **Content-based filtering** with TF-IDF and cosine similarity (Parts 1-2)
2. **Collaborative filtering** with a small user-job ratings table (Part 3)
3. **BM25 ranking**, compared against the TF-IDF/cosine baseline (Part 4)

Part 5 asks you to compare the three approaches and apply them to a new scenario.


## Learning goals

- Explain the difference between content-based and collaborative filtering.
- Represent text descriptions using TF-IDF and interpret the resulting feature matrix.
- Calculate and interpret cosine similarity for a content-based recommendation.
- Compare users with sparse rating data and produce one collaborative-filtering recommendation.
- Apply BM25 to rank job descriptions against a search query, using the `rank-bm25` library.
- Compare BM25 and TF-IDF/cosine rankings on the same query and explain why they differ.
- Compare recommendation methods and recognize their limitations.

## Setup

Run the next cell once. It installs `rank-bm25`, the only package Colab does not already include, using the same guarded pattern as the Week 06 notebook (check first, install quietly if missing). Everything else in this notebook uses `pandas`, `numpy`, and `scikit-learn`, which Colab already provides.

In [ ]:
import importlib.util
import subprocess
import sys

# One-time setup. rank-bm25 is small and pure Python; this is the only
# install in the notebook. Safe to rerun.
if importlib.util.find_spec("rank_bm25") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank-bm25"])


In [ ]:
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42


## 1. Content-based filtering: inspect the job collection

The job collection below extends the six jobs from the reading (`J1`-`J6`) with six new ones (`J7`-`J12`), still fit for Alex to search. Some are strong matches for Alex's early-career Python/SQL reporting goal; some look similar on the surface but require senior or management experience the reading already warned about; two are unrelated maintenance jobs included as a sanity check (an unrelated job should score near zero).

In [ ]:
jobs = pd.DataFrame([
    {"job_id": "J1", "title": "Junior Data Analyst",
     "description": "Early-career data analyst role. Use Python and SQL to build recurring reports for the operations team. New graduates welcome; on-the-job training provided."},
    {"job_id": "J2", "title": "Analytics Director",
     "description": "Lead the analytics strategy using Python and SQL. Requires ten or more years of experience managing a data team and setting department priorities."},
    {"job_id": "J3", "title": "Research Data Assistant",
     "description": "Support a research group with Python and SQL reporting on language data. NLP training will be provided; no prior NLP experience required."},
    {"job_id": "J4", "title": "SQL Reporting Analyst",
     "description": "Build SQL database reports and apply basic text tagging to customer feedback. Early-career role with a supportive onboarding program."},
    {"job_id": "J5", "title": "Senior NLP Engineer",
     "description": "Build and deploy Python language models in production. Requires several years of hands-on experience shipping NLP systems at scale."},
    {"job_id": "J6", "title": "Equipment Maintenance Technician",
     "description": "Inspect and maintain manufacturing machinery. Mechanical aptitude required; no programming or data experience needed."},
    {"job_id": "J7", "title": "Junior SQL Analyst",
     "description": "Entry-level analyst writing SQL queries and building dashboards. No management responsibilities; ideal for a first data role."},
    {"job_id": "J8", "title": "Data Science Intern",
     "description": "Learn Python, Python, and more Python while supporting the data science team with small reporting tasks. Early-career, mentorship-focused internship."},
    {"job_id": "J9", "title": "Senior Project Manager",
     "description": "Manage cross-functional technical projects. Some familiarity with Python is helpful, but the role is centered on people management and long-term planning."},
    {"job_id": "J10", "title": "NLP Research Scientist",
     "description": "Conduct advanced NLP research with large language models. PhD or equivalent research experience required; focus on publishing novel methods."},
    {"job_id": "J11", "title": "Business Intelligence Reporting Analyst",
     "description": "Maintain SQL-based reporting dashboards for business stakeholders. Early-career friendly, with limited opportunity to grow into modeling work."},
    {"job_id": "J12", "title": "Warehouse Equipment Technician",
     "description": "Perform routine maintenance on warehouse equipment. Hands-on mechanical work; no data or text skills involved."},
]).set_index("job_id")

jobs


Alex's profile is one short piece of text, written the same way a job description is written, so it can be compared using the same vocabulary and weighting.

In [ ]:
alex_profile = (
    "Early-career candidate skilled in Python and SQL, interested in reporting roles. "
    "Open to learning NLP but does not want a senior management position."
)
alex_profile


### TODO 1: Fit TF-IDF on the job collection

Fit a `TfidfVectorizer` on the `description` column of `jobs`, then transform Alex's profile into the *same* vector space (do not re-fit on the profile).

In [ ]:
# TODO 1a: create a TfidfVectorizer. stop_words="english" drops common
# English words (like "the" and "and") before comparing descriptions.
# Try:  TfidfVectorizer(stop_words="english")
tfidf = ...

# TODO 1b: tfidf has not seen any text yet. fit_transform() both learns
# the vocabulary from the jobs AND converts them to vectors in one step.
# Try:  tfidf.fit_transform(jobs["description"])
job_tfidf = ...

# TODO 1c: tfidf already learned its vocabulary above, from the jobs.
# transform() reuses that same vocabulary without learning a new one.
# Try:  tfidf.transform([alex_profile])
alex_tfidf = ...

print("Job feature matrix shape:", job_tfidf.shape)
print("Alex profile vector shape:", alex_tfidf.shape)


**Guided question:** The feature matrix has one row per job and one column per vocabulary term. Why must Alex's profile be *transformed* using the vectorizer already fit on the jobs, rather than fitting a brand-new vectorizer on the profile text alone?

## 2. Cosine similarity and a recommendation function

Now compare Alex's profile vector against every job vector and rank the jobs.

### TODO 2: Compute cosine similarity and rank the jobs

In [ ]:
# TODO 2a: cosine_similarity(A, B) compares every row of A against every
# row of B and returns a 2D array. We only have one profile, so flatten()
# turns that single row into a plain 1D array of scores, one per job.
# Try:  cosine_similarity(alex_tfidf, job_tfidf).flatten()
alex_scores = ...

scored_jobs = jobs.copy()
# TODO 2b: attach the scores you just computed as a new column.
scored_jobs["cosine_score"] = ...
scored_jobs = scored_jobs.sort_values("cosine_score", ascending=False)
scored_jobs[["title", "cosine_score"]]


### TODO 3: Build a recommendation function

Write a function that returns the top-`k` jobs for a given profile vector, excluding nothing special here (there is no "query item" to exclude, since Alex's profile is not itself a job).

In [ ]:
def recommend_jobs(profile_vector, job_matrix, jobs_df, k=5):
    # This function repeats exactly what you just did in TODO 2, packaged
    # so it can be reused for any profile vector, not only Alex's.

    # TODO 3a: same pattern as TODO 2a, but with profile_vector/job_matrix
    # instead of alex_tfidf/job_tfidf.
    # Try:  cosine_similarity(profile_vector, job_matrix).flatten()
    scores = ...

    ranked = jobs_df.copy()
    ranked["cosine_score"] = scores

    # TODO 3b: sort descending (best match first) and keep only the top k
    # rows, showing just the title and score.
    # Try:  ranked.sort_values("cosine_score", ascending=False).head(k)[["title", "cosine_score"]]
    return ...


recommend_jobs(alex_tfidf, job_tfidf, jobs, k=5)


**Guided questions:**

- Why might two jobs have high cosine similarity even though one is clearly a poor match for Alex (hint: look at `J2` and `J9`)?
- Does high text similarity guarantee a good recommendation?
- What information does TF-IDF miss that a human reader would notice immediately in the full description?

## 3. Collaborative filtering

Now set the job *text* aside. A small group of users has rated a subset of these jobs (1-5 scale). A blank cell means the user has not rated that job, **not** that they dislike it.

In [ ]:
rating_job_ids = ["J1", "J2", "J3", "J4", "J5", "J7", "J9", "J10", "J11"]

ratings = pd.DataFrame(
    {
        "J1":  [5, 5, np.nan, np.nan, 3],
        "J2":  [np.nan, np.nan, np.nan, 5, 2],
        "J3":  [4, 4, 5, np.nan, np.nan],
        "J4":  [4, 5, np.nan, np.nan, 3],
        "J5":  [np.nan, np.nan, 4, 3, np.nan],
        "J7":  [5, 4, np.nan, np.nan, 3],
        "J9":  [np.nan, np.nan, 2, 5, np.nan],
        "J10": [np.nan, np.nan, 5, np.nan, np.nan],
        "J11": [np.nan, 5, np.nan, np.nan, np.nan],
    },
    index=["Alex", "Blair", "Casey", "Dana", "Evan"],
)[rating_job_ids]

ratings


### Predict before computing

Skim the table. Which user do you expect to be most similar to Alex, and why? Which jobs do you expect to be hard to compare because too few users rated them?

### TODO 4: Compare users with cosine similarity

Missing ratings are filled with 0 only so the similarity calculation can run. This is a simplification, not a claim that a missing rating means the lowest possible score. Compute Alex's cosine similarity to every other user.

In [ ]:
ratings_filled = ratings.fillna(0)

alex_vector = ratings_filled.loc[["Alex"]]
other_users = ratings_filled.drop(index="Alex")

# TODO 4a: this is the same cosine_similarity(...).flatten() pattern from
# Part 2, just on rating rows instead of TF-IDF rows.
# Try:  cosine_similarity(alex_vector, other_users).flatten()
similarity_scores = ...

# TODO 4b: wrap similarity_scores in a pandas Series indexed by the other
# users' names, then sort it so the most similar user comes first.
# Try:  pd.Series(similarity_scores, index=other_users.index).sort_values(ascending=False)
user_similarity = ...
user_similarity


### TODO 5: Make one collaborative-filtering recommendation

Using Alex's most similar user, recommend a job Alex has **not** yet rated that the neighbor rated highly.

In [ ]:
most_similar_user = user_similarity.index[0]
print("Most similar user to Alex:", most_similar_user, f"(cosine = {user_similarity.iloc[0]:.3f})")

alex_rated = ratings.loc["Alex"].dropna().index

# TODO 5a: neighbor_ratings should hold only the jobs most_similar_user
# actually rated (drop their own missing values, the same way ratings for
# any single user could be read).
# Try:  ratings.loc[most_similar_user].dropna()
neighbor_ratings = ...

# TODO 5b: a recommendation should only point to something Alex hasn't
# already rated. Remove any job id in alex_rated from neighbor_ratings.
# errors="ignore" means it is fine if some of those ids are not present.
# Try:  neighbor_ratings.drop(index=alex_rated, errors="ignore")
candidate_jobs = ...

if len(candidate_jobs) > 0:
    # TODO 5c: among candidate_jobs, find the job_id with the highest
    # rating. sort_values(ascending=False).index[0] is the same pattern
    # used elsewhere in this notebook to find a top result.
    recommended_job_id = ...
    print(f"Recommend {recommended_job_id} ({jobs.loc[recommended_job_id, 'title']}): "
          f"{most_similar_user} rated it {neighbor_ratings[recommended_job_id]:.0f}/5, "
          "and Alex has not rated it.")
else:
    print("No unseen job to recommend from this neighbor; their rated jobs overlap entirely with Alex's.")


**Guided questions:**

- How is this different from the content-based approach in Parts 1-2? What evidence does it use that TF-IDF never sees?
- What happens when a new user has no ratings at all? Could this method make a recommendation for them?
- The blank cells were filled with 0 to run the calculation. What could go wrong if 0 is later read as "this user dislikes the job," instead of "we don't know"?

## 4. BM25 ranking and a direct comparison with TF-IDF/cosine

BM25 ranks documents against a search *query*, rather than a profile paragraph. Use the same 12 jobs and compare BM25's ranking to a TF-IDF/cosine ranking of the exact same query.

In [ ]:
query = "early career Python SQL reporting role"


def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())


tokenized_jobs = [tokenize(desc) for desc in jobs["description"]]
tokenized_query = tokenize(query)
tokenized_jobs[:2]


### TODO 6: Compute BM25 scores

Use `BM25Okapi` from `rank-bm25` on the tokenized job collection, then score the tokenized query against it. You are not implementing the BM25 formula yourself; the library handles term frequency, inverse document frequency, and document-length normalization. Elasticsearch's documented defaults are `k1 = 1.2` and `b = 0.75`; `rank-bm25` uses the same defaults unless you pass different values.

In [ ]:
# TODO 6a: BM25Okapi(tokenized_jobs) builds the BM25 index over the job
# collection, the same way TfidfVectorizer().fit_transform(...) built the
# TF-IDF index in Part 1.
# Try:  BM25Okapi(tokenized_jobs)
bm25 = ...

# TODO 6b: get_scores(...) scores every job in the index against one
# tokenized query and returns one score per job.
# Try:  bm25.get_scores(tokenized_query)
bm25_scores = ...

bm25_ranked = jobs.copy()
bm25_ranked["bm25_score"] = bm25_scores
bm25_ranked = bm25_ranked.sort_values("bm25_score", ascending=False)
bm25_ranked[["title", "bm25_score"]]


### TODO 7: Rank the same query with TF-IDF/cosine, side by side

Transform `query` through the **same** `tfidf` vectorizer from Part 1 (do not re-fit), score it against `job_tfidf`, and place the two rankings next to each other for comparison.

In [ ]:
# TODO 7a: transform (not fit) `query` with the SAME tfidf vectorizer
# from Part 1, the same reasoning as TODO 1c.
# Try:  tfidf.transform([query])
query_tfidf = ...

# TODO 7b: same cosine_similarity(...).flatten() pattern as TODO 2a/4a,
# this time scoring query_tfidf against job_tfidf.
# Try:  cosine_similarity(query_tfidf, job_tfidf).flatten()
tfidf_query_scores = ...

comparison = jobs.copy()
comparison["bm25_score"] = bm25_scores
comparison["tfidf_cosine_score"] = tfidf_query_scores

# This first line is done for you: top 5 jobs by BM25 score.
bm25_top5 = comparison.sort_values("bm25_score", ascending=False).head(5)[["title", "bm25_score"]]

# TODO 7c: mirror the line above, but sorted by "tfidf_cosine_score"
# instead, showing ["title", "tfidf_cosine_score"].
cosine_top5 = ...

print("Top 5 by BM25:")
display(bm25_top5)
print("\nTop 5 by TF-IDF/cosine:")
display(cosine_top5)


**Guided questions:**

- Where do the two rankings agree? Where do they put jobs in a different order?
- `J8` repeats the word "Python" three times in a short description. Look at how `J8` scores under BM25 versus TF-IDF/cosine. Does repeating a term help the same way under both methods?
- How does document length affect ranking? Compare a short job description to a longer one that mentions the same query terms about the same number of times.
- What happens if the query uses a synonym that is absent from a job description word-for-word (for example, "reporting" instead of a description that only says "dashboards")? Would either method catch that match?

## 5. Reflection and comparison

Fill in the table below.

| Method | Input Data | Strength | Limitation |
|---|---|---|---|
| Content-based (TF-IDF + cosine) | Job text and a profile written the same way | ? | ? |
| Collaborative filtering | User-job ratings (no text at all) | ? | ? |
| BM25 | A search query and the job text collection | ? | ? |

**Scenario questions:**

1. A brand-new user signs up with no ratings yet, but writes two sentences about what kind of role they want. Which method(s) from this notebook could make a recommendation for them right now, and which could not?
2. Alex later says: "Show me jobs most like J11, not jobs that match my profile." Which part of this notebook already computes something close to that request, and what would you need to change to answer it directly?

*Conceptual extension (not implemented here):* nearest-neighbor search (KNN) uses the same similarity idea as Part 2 and Part 3 to retrieve the closest items or users at scale, rather than scoring every item one at a time. Everything in this notebook is a small enough collection that a full comparison already works, which is itself why a dedicated KNN step was not needed.

## Validation checks

These checks confirm the notebook ran top to bottom with no hidden state.

In [ ]:
assert len(jobs) == 12, "Expected 12 jobs in the collection."
assert job_tfidf.shape[0] == 12
assert len(recommend_jobs(alex_tfidf, job_tfidf, jobs, k=5)) == 5
assert ratings.shape == (5, 9), "Expected 5 users and 9 rated jobs."
assert len(bm25_scores) == 12
assert len(tfidf_query_scores) == 12
print("All checks passed.")
